In [ ]:
from curl_cffi import requests
from bs4 import BeautifulSoup
from urllib.parse import urljoin
import time
import random
import re
import pandas as pd
from google.colab import files
import os

print("🚀 KHỞI ĐỘNG BOT: TRANG 26 VỚI COOKIE MỚI NHẤT...")

BASE = "https://wikicv.net"
# ĐÃ CHỈNH START=500 ĐỂ CHẠY ĐÚNG TRANG 26
START_URL = "https://wikicv.net/tim-kiem?qs=1&gender=5794f03dd7ced228f4419198&m=3&q=&start=500&so=1&y=2026&vo=1"

# COOKIE MỚI NHẤT SẾP VỪA GỬI (ĐOẠN cto_bundle=JhRBhV...)
cookie_cua_chi = '_ga=GA1.1.1009058345.1771417284; _uidcms=1771417286214481472; __gsas=ID=854d657c499ed105:T=1771417341:RT=1771417341:S=ALNI_MZeY_TeGBq5GMbyjaa-ledLOVMuQA; _clck=1dsp33s%5E2%5Eg3o%5E0%5E2240; _ga_8R28QVTQNJ=GS2.1.s1771417349$o1$g1$t1771418918$j29$l0$h0; _ga_XFX33MFDHM=GS2.1.s1771418508$o1$g1$t1771419524$j44$l0$h0; _ga_7NXWGWHXCV=GS2.1.s1771668027$o1$g1$t1771668057$j30$l0$h0; express.sid="s:si8xsqyb0fXFV_7r6n2Hdp9Q6ivx-i4W.OIbYkgk9EOxIwIOFQtG6QqPWwzFpsBSGCYNB/89f6W4"; __RC=4; __UF=-1; __R=1; __tb=0; cto_bundle=JhRBhV9TQUxFaHN5YUV3QnFuYTJWZzd4ZTNNRWZFR3ZpM2x1eUhaMUxnMThDNm5ZbWVRNjRIRUIlMkZqUjRma0xhZCUyRjB5OXZ3cXh5c24ycFAyaGtycjMlMkZJcU9nV1FPczlYc0FzRjh1UGRNUUVFTEpXN1dvN0poMmc1eHJFWmxwbnY4YTglMkJFNVVjV2p4YUFhZlY3VVI1RFVDemhnZyUzRCUzRA; bs_onshow=1; rigelcdp-session-id=12e35a2e-3404-4c8e-b424-b38f956a48d7; __tr_geo={%22country%22:{%22name%22:%22Vietnam%22%2C%22code%22:%22VN%22}%2C%22city%22:%22Hanoi%22}; __gads=ID=f0c111b12746b269:T=1773759966:RT=1773895950:S=ALNI_MZ13xpwkmjYH3eJCUteeTo1g-xJrA; __gpi=UID=000012218b3eacaa:T=1773759966:RT=1773895950:S=ALNI_MZhlla3_YwAsAaji90iGqIs2nj6cQ; __eoi=ID=8c86f34769843120:T=1773759966:RT=1773895950:S=AA-AfjZ-mjcPrMBtF-blIMmJOjN-; FCCDCF=%5Bnull%2Cnull%2Cnull%2Cnull%2Cnull%2Cnull%2C%5B%5B32%2C%22%5B%5C%2296c35a21-7536-4d85-88ec-c2a47483291f%5C%22%2C%5B1773759962%2C384000000%5D%5D%22%5D%5D%5D; __uif=__uid%3A4872869511907961728%7C__create%3A1771417286; FCNEC=%5B%5B%22AKsRol9F1oIwYwuRRE64IX-HqLRCjP6MipYIXJRnkJgiYX8gR6J5QPjUyCJjNv_Oq-qlUSkVS74pcWr0Hzw0sry8E4WihtLRnQux7-Lp_mRMIF9tNfy2AxWHTdmshIS0yGiinmQ3eWVf3ggT70NEqHiwnid_UGrz0Q%3D%3D%22%5D%5D; _ga_DSEYM6VX97=GS2.1.s1773895943$o6$g1$t1773895989$j60$l0$h0; _ga_YCSZZTM0SE=GS2.1.s1773895943$o6$g1$t1773895989$j60$l0$h0; _ga_SXHZQVHX0C=GS2.1.s1773895943$o6$g1$t1773895989$j60$l0$h0; _ga_XBF5L66EJ2=GS2.1.s1773895946$o4$g1$t1773895989$j17$l1$h1445935044'

headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/137.0.0.0 Safari/537.36",
    "Cookie": cookie_cua_chi
}

session = requests.Session(impersonate="chrome110", timeout=25)

def parse_abbr_num(text):
    if not text: return 0
    text = str(text).strip().lower().replace(",", "").replace(" ", "")
    m = re.search(r"(\d+(?:\.\d+)?)([mk]?)", text)
    if not m: return 0
    num, suffix = float(m.group(1)), m.group(2)
    if suffix == "k": return int(num * 1000)
    if suffix == "m": return int(num * 1000000)
    return int(num)

def extract_so_chuong(detail_soup, toan_bo_chu):
    for tag in detail_soup.find_all(['p', 'div', 'li', 'span']):
        txt = tag.get_text(" ", strip=True).lower()
        if txt.startswith("mới nhất:"):
            nums = re.findall(r'\d+', txt)
            if nums: return int(nums[-1])

    patterns = [r'(\d+)\s*chương\s*chính\s*văn', r'(\d+)\s*chương', r'hoàn\s*(\d+)', r'phần\s*(\d+)']
    for p in patterns:
        m = re.search(p, toan_bo_chu.lower())
        if m: return int(m.group(1))
    return 0

def get_soup(url):
    resp = session.get(url, headers=headers)
    return BeautifulSoup(resp.content, "html.parser")

def get_book_links(soup):
    links = []
    for a in soup.find_all("a", href=True):
        href = a["href"]
        if "/truyen/" in href and "khu-tim-truyen" not in href.lower():
            links.append(urljoin(BASE, href))
    return list(dict.fromkeys(links))

def find_next_page_url(soup, current_page):
    target = str(current_page + 1)
    for a in soup.find_all("a", href=True):
        if a.get_text(" ", strip=True) == target:
            return urljoin(BASE, a["href"])
    return None

# --- VÒNG LẶP CHÍNH ---
url = START_URL
data_list = []
file_backup = "Data_BachHop_Trang26_Backup.csv"

try:
    for page in range(26, 201):
        print(f"\n=== 📑 ĐANG QUÉT TRANG {page} | Đã lấy: {len(data_list)} truyện ===")
        soup = get_soup(url)
        links = get_book_links(soup)

        if not links:
            print("\n⚠️ Không thấy link! Kiểm tra Cookie hoặc đổi mạng (4G).")
            break

        for book_url in links:
            try:
                time.sleep(random.uniform(2.2, 3.8))
                b_soup = get_soup(book_url)
                toan_bo_chu = b_soup.get_text(" ", strip=True)

                h_tag = b_soup.find('h2') or b_soup.find('h1')
                ten = h_tag.text.strip() if h_tag else "N/A"
                if "TÌM TRUYỆN" in ten.upper(): continue

                tac_gia, tinh_trang, the_loai = "Ẩn danh", "Đang cập nhật", "N/A"
                for tag_element in b_soup.find_all(['p', 'div', 'li']):
                    t_text = tag_element.get_text(" ", strip=True)
                    if t_text.startswith("Tác giả:"): tac_gia = t_text.replace("Tác giả:", "").strip()
                    elif t_text.startswith("Tình trạng:"): tinh_trang = t_text.replace("Tình trạng:", "").strip()
                    elif t_text.startswith("Thể loại:"): the_loai = t_text.replace("Thể loại:", "").strip()

                so_chuong = extract_so_chuong(b_soup, toan_bo_chu)
                stats = [tag.get_text(strip=True) for tag in b_soup.select("span[data-ready='abbrNum']")]
                vxem = parse_abbr_num(stats[0]) if len(stats) > 0 else 0
                sao = parse_abbr_num(stats[1]) if len(stats) > 1 else 0
                bluan = parse_abbr_num(stats[2]) if len(stats) > 2 else 0

                cam_on = 0
                if "Cảm ơn:" in toan_bo_chu:
                    try:
                        cam_on_text = toan_bo_chu.split("Cảm ơn:")[1].split('lần')[0].strip()
                        cam_on = parse_abbr_num(cam_on_text)
                    except: pass

                data_list.append({
                    "Trang": page, "Ten_Truyen": ten, "Tac_Gia": tac_gia,
                    "The_Loai": the_loai, "Tinh_Trang": tinh_trang, "So_Chuong": so_chuong,
                    "Luot_Xem": vxem, "Luot_Sao": sao, "Luot_Binh_Luan": bluan,
                    "Luot_Cam_On": cam_on, "Link": book_url
                })
                print(f"   ✅ [Tổng: {len(data_list)}] {ten[:20]}...")

            except KeyboardInterrupt: raise
            except: continue

        if data_list:
            pd.DataFrame(data_list).to_csv(file_backup, index=False, encoding='utf-8-sig')

        next_url = find_next_page_url(soup, page)
        if not next_url: break
        url = next_url

except KeyboardInterrupt:
    print("\n👋 Đã dừng ")
finally:
    if data_list:
        file_name = f"Data_BachHop_Trang26_CookieMoi_Xong_{len(data_list)}.csv"
        if os.path.exists(file_backup): os.rename(file_backup, file_name)
        files.download(file_name)

🚀 KHỞI ĐỘNG BOT: TRANG 26 VỚI COOKIE MỚI NHẤT...

=== 📑 ĐANG QUÉT TRANG 26 | Đã lấy: 0 truyện ===
   ✅ [Tổng: 1] Hỏi đáp buông xuống,...
   ✅ [Tổng: 2] Ta thế nhưng xuyên q...
   ✅ [Tổng: 3] Ahri kỳ diệu mạo hiể...
   ✅ [Tổng: 4] Sư tôn, ta này kịch ...
   ✅ [Tổng: 5] Nữ tướng quân cùng t...
   ✅ [Tổng: 6] Nữ chủ đại nhân, ta ...
   ✅ [Tổng: 7] Đại sư tỷ nàng không...
   ✅ [Tổng: 8] Nam chủ hậu viện chá...
   ✅ [Tổng: 9] Lão tử chính là Alic...
   ✅ [Tổng: 10] Ta thật không phải q...
   ✅ [Tổng: 11] Cái gì! Ngươi muốn t...
   ✅ [Tổng: 12] Mạt thế chi lén đi G...
   ✅ [Tổng: 13] Xuyên đến tra A tra ...
   ✅ [Tổng: 14] Xuyên qua sau ta sao...
   ✅ [Tổng: 15] Xuyên thư cưới ly hô...
   ✅ [Tổng: 16] Ta, Shiba Mayuki, là...
   ✅ [Tổng: 17] Xuyên thư: Cùng ảnh ...
   ✅ [Tổng: 18] Xuyên thành luyến ái...
   ✅ [Tổng: 19] Giả Thái Tử thế thân...
   ✅ [Tổng: 20] Bạn gái cũ hiểu biết...

=== 📑 ĐANG QUÉT TRANG 27 | Đã lấy: 20 truyện ===
   ✅ [Tổng: 21] Cổ đại nữ nhân thật ...
   ✅ [Tổng: 22] Phò m

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
from curl_cffi import requests
from bs4 import BeautifulSoup
from urllib.parse import urljoin
import time
import random
import re
import pandas as pd
from google.colab import files
import os

print("🚀 KHỞI ĐỘNG BOT: CHIẾN DỊCH QUÉT TRUYỆN NGÔN TÌNH (TỪ TRANG 1)...")

BASE = "https://wikicv.net"
# LINK MỚI SẾP GỬI (Chuyên mục Ngôn tình)
START_URL = "https://wikicv.net/tim-kiem?qs=1&gender=5794f03dd7ced228f4419195&tc=&tf=0&m=3&y=2026&q="

# COOKIE MỚI NHẤT SẾP GỬI (ĐÃ CẬP NHẬT)
cookie_cua_chi = '_ga=GA1.1.442769808.1773887332; _uidcms=1773887332953852475; cto_bundle=fPD-cV9yR3pBSmdEYVUwTTN1dHdoRXh1NmU5QWJnTGklMkJJWkE3UVZ1eFdIMTJzZDc1bW1pSG5IVk1FdXpUQWppdDNwSWJGRHpGTUNMMFYzdTJVQXlqMlVOOEtzWjNFMDY0MmY3TThVV3VYbHNoRmVENFdxJTJGRjIxMHcycmoxRUljT09tb1NEJTJCJTJCJTJCQWRpcjFBMFd4N0xXclc0UjBnJTNEJTNE; _ga_7NXWGWHXCV=GS2.1.s1773887332$o1$g0$t1773887335$j57$l0$h0; rigelcdp-session-id=8e64f9b0-7991-415f-84f6-4e65f3e607e4; __tr_geo={%22country%22:{%22name%22:%22Vietnam%22%2C%22code%22:%22VN%22}%2C%22city%22:%22Hanoi%22}; express.sid=s:oEJW8jdeClGAeWB1tMurUtc1zrd-0Wb4.j2Fwc5J5PWI1GEhW7lI6eypCS53SoJuniRtJQiy8M5c; __RC=4; __UF=-1; __R=1; __tb=0; bs_onshow=1; __gads=ID=e492ecf20d8482a5:T=1773887338:RT=1773905886:S=ALNI_MYYXo_kg_HDx_sI6k_yvtM7GP-Iaw; __gpi=UID=00001223e40a5906:T=1773887338:RT=1773905886:S=ALNI_MajvRAJ7uRftoNmJdE778fjfMNqQQ; __eoi=ID=7839beeb7dcd9b91:T=1773887338:RT=1773905886:S=AA-AfjZB0IClVfusRkz02d0GhSJ6; FCCDCF=%5Bnull%2Cnull%2Cnull%2Cnull%2Cnull%2Cnull%2C%5B%5B32%2C%22%5B%5C%22bb90ca49-aede-4744-a49a-9ad04c5b1d81%5C%22%2C%5B1773887332%2C638000000%5D%5D%22%5D%5D%5D; __uif=__uid%3A2687333658712079130%7C__create%3A1773887333; FCNEC=%5B%5B%22AKsRol_SZYg9SrYZiPEokCG0SM-7Q3nk0fGPAO3SI40r1NYsP1eUxCj_75gZII_NlpUDgO_dcX2OFHwpoAHHlomcttivC4zOdwViNlI8DuMEcPXFvZDOjjl6xvRIvfAWXjfQQbHM85ZbchMhRDwCOXCh_pRK3iVwjQ%3D%3D%22%5D%5D; _ga_YCSZZTM0SE=GS2.1.s1773905878$o3$g1$t1773905908$j45$l0$h0; _ga_SXHZQVHX0C=GS2.1.s1773905878$o3$g1$t1773905908$j45$l0$h0; _ga_DSEYM6VX97=GS2.1.s1773905878$o3$g1$t1773905908$j45$l0$h0; _ga_XBF5L66EJ2=GS2.1.s1773905881$o3$g1$t1773905908$j33$l1$h2037671551'

headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/137.0.0.0 Safari/537.36",
    "Cookie": cookie_cua_chi
}

session = requests.Session(impersonate="chrome110", timeout=25)

def parse_abbr_num(text):
    if not text: return 0
    text = str(text).strip().lower().replace(",", "").replace(" ", "")
    m = re.search(r"(\d+(?:\.\d+)?)([mk]?)", text)
    if not m: return 0
    num, suffix = float(m.group(1)), m.group(2)
    if suffix == "k": return int(num * 1000)
    if suffix == "m": return int(num * 1000000)
    return int(num)

def extract_so_chuong(detail_soup, toan_bo_chu):
    for tag in detail_soup.find_all(['p', 'div', 'li', 'span']):
        txt = tag.get_text(" ", strip=True).lower()
        if txt.startswith("mới nhất:"):
            nums = re.findall(r'\d+', txt)
            if nums: return int(nums[-1])

    patterns = [r'(\d+)\s*chương\s*chính\s*văn', r'(\d+)\s*chương', r'hoàn\s*(\d+)', r'phần\s*(\d+)']
    for p in patterns:
        m = re.search(p, toan_bo_chu.lower())
        if m: return int(m.group(1))
    return 0

def get_soup(url):
    resp = session.get(url, headers=headers)
    return BeautifulSoup(resp.content, "html.parser")

def get_book_links(soup):
    links = []
    for a in soup.find_all("a", href=True):
        href = a["href"]
        if "/truyen/" in href and "khu-tim-truyen" not in href.lower():
            links.append(urljoin(BASE, href))
    return list(dict.fromkeys(links))

def find_next_page_url(soup, current_page):
    target = str(current_page + 1)
    for a in soup.find_all("a", href=True):
        if a.get_text(" ", strip=True) == target:
            return urljoin(BASE, a["href"])
    return None

# --- VÒNG LẶP CHÍNH ---
url = START_URL
data_list = []
file_backup = "Data_NgonTinh_Backup.csv"

try:
    for page in range(1, 201):
        print(f"\n=== 📑 ĐANG QUÉT TRANG {page} | Đã lấy: {len(data_list)} truyện ===")
        soup = get_soup(url)
        links = get_book_links(soup)

        if not links:
            print("\n⚠️ Không thấy link! Kiểm tra Cookie hoặc đổi IP nếu bị chặn.")
            break

        for book_url in links:
            try:
                time.sleep(random.uniform(2.5, 4.0))
                b_soup = get_soup(book_url)
                toan_bo_chu = b_soup.get_text(" ", strip=True)

                h_tag = b_soup.find('h2') or b_soup.find('h1')
                ten = h_tag.text.strip() if h_tag else "N/A"
                if "TÌM TRUYỆN" in ten.upper(): continue

                tac_gia, tinh_trang, the_loai = "Ẩn danh", "Đang cập nhật", "N/A"
                for tag_element in b_soup.find_all(['p', 'div', 'li']):
                    t_text = tag_element.get_text(" ", strip=True)
                    if t_text.startswith("Tác giả:"): tac_gia = t_text.replace("Tác giả:", "").strip()
                    elif t_text.startswith("Tình trạng:"): tinh_trang = t_text.replace("Tình trạng:", "").strip()
                    elif t_text.startswith("Thể loại:"): the_loai = t_text.replace("Thể loại:", "").strip()

                so_chuong = extract_so_chuong(b_soup, toan_bo_chu)
                stats = [tag.get_text(strip=True) for tag in b_soup.select("span[data-ready='abbrNum']")]
                vxem = parse_abbr_num(stats[0]) if len(stats) > 0 else 0
                sao = parse_abbr_num(stats[1]) if len(stats) > 1 else 0
                bluan = parse_abbr_num(stats[2]) if len(stats) > 2 else 0

                cam_on = 0
                if "Cảm ơn:" in toan_bo_chu:
                    try:
                        cam_on_text = toan_bo_chu.split("Cảm ơn:")[1].split('lần')[0].strip()
                        cam_on = parse_abbr_num(cam_on_text)
                    except: pass

                data_list.append({
                    "Trang": page, "Ten_Truyen": ten, "Tac_Gia": tac_gia,
                    "The_Loai": the_loai, "Tinh_Trang": tinh_trang, "So_Chuong": so_chuong,
                    "Luot_Xem": vxem, "Luot_Sao": sao, "Luot_Binh_Luan": bluan,
                    "Luot_Cam_On": cam_on, "Link": book_url
                })
                print(f"   ✅ [Tổng: {len(data_list)}] {ten[:20]}...")

                # Tự bảo vệ: Nếu đạt 500 truyện thì lưu ngay để tránh bị ban mất hết
                if len(data_list) % 500 == 0:
                    pd.DataFrame(data_list).to_csv(file_backup, index=False, encoding='utf-8-sig')
                    print(f"💾 Đã backup tạm thời {len(data_list)} truyện.")

            except KeyboardInterrupt: raise
            except: continue

        if data_list:
            pd.DataFrame(data_list).to_csv(file_backup, index=False, encoding='utf-8-sig')

        next_url = find_next_page_url(soup, page)
        if not next_url: break
        url = next_url

except KeyboardInterrupt:
    print("\n👋 Đã dừng ")
finally:
    if data_list:
        file_name = f"Data_NgonTinh_Xong_{len(data_list)}_Truyen.csv"
        if os.path.exists(file_backup): os.rename(file_backup, file_name)
        files.download(file_name)

🚀 KHỞI ĐỘNG BOT: CHIẾN DỊCH QUÉT TRUYỆN NGÔN TÌNH (TỪ TRANG 1)...

=== 📑 ĐANG QUÉT TRANG 1 | Đã lấy: 0 truyện ===
   ✅ [Tổng: 1] Thiên Y Phượng Cửu (...
   ✅ [Tổng: 2] Mị vương sủng thê: Q...
   ✅ [Tổng: 3] Xuyên nhanh: Nữ xứng...
   ✅ [Tổng: 4] Tà Vương truy thê / ...
   ✅ [Tổng: 5] Tuyệt thế thần y: Ph...
   ✅ [Tổng: 6] Chí tôn đồng thuật s...
   ✅ [Tổng: 7] Nông môn trưởng tỷ c...
   ✅ [Tổng: 8] Bạo sủng cuồng thê: ...
   ✅ [Tổng: 9] Hắc hóa nam chủ tổng...
   ✅ [Tổng: 10] Nông gia tiểu phúc n...
   ✅ [Tổng: 11] Chớ chọc phúc hắc cu...
   ✅ [Tổng: 12] Một thai nhị bảo: Hà...
   ✅ [Tổng: 13] Trở về 60: Làm ruộng...
   ✅ [Tổng: 14] Xuyên nhanh: Cái này...
   ✅ [Tổng: 15] Phế sài muốn nghịch ...
   ✅ [Tổng: 16] Quỷ đế cuồng thê: Ăn...
   ✅ [Tổng: 17] Ẩn hôn ngọt sủng: Đạ...
   ✅ [Tổng: 18] Xuyên nhanh: Nữ chủ ...
   ✅ [Tổng: 19] Mau xuyên: Vai ác lạ...
   ✅ [Tổng: 20] Trọng sinh 80: Tức p...

=== 📑 ĐANG QUÉT TRANG 2 | Đã lấy: 20 truyện ===
   ✅ [Tổng: 21] Y phẩm độc phi khuyn...
   ✅ [

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
from curl_cffi import requests
from bs4 import BeautifulSoup
from urllib.parse import urljoin
import time
import random
import re
import pandas as pd
from google.colab import files
import os

print("🚀 TIẾP TỤC QUÉT NGÔN TÌNH VỚI COOKIE MỚI...")

BASE = "https://wikicv.net"
# Link trang 26 hoặc trang sếp đang quét dở
START_URL = "https://wikicv.net/tim-kiem?qs=1&gender=5794f03dd7ced228f4419195&m=3&q=&start=500&so=1&y=2026&vo=1"

# COOKIE MỚI NHẤT SẾP VỪA GỬI
cookie_cua_chi = '_ga=GA1.1.1009058345.1771417284; _uidcms=1771417286214481472; __gsas=ID=854d657c499ed105:T=1771417341:RT=1771417341:S=ALNI_MZeY_TeGBq5GMbyjaa-ledLOVMuQA; _clck=1dsp33s%5E2%5Eg3o%5E0%5E2240; _ga_8R28QVTQNJ=GS2.1.s1771417349$o1$g1$t1771418918$j29$l0$h0; _ga_XFX33MFDHM=GS2.1.s1771418508$o1$g1$t1771419524$j44$l0$h0; _ga_7NXWGWHXCV=GS2.1.s1771668027$o1$g1$t1771668057$j30$l0$h0; express.sid="s:si8xsqyb0fXFV_7r6n2Hdp9Q6ivx-i4W.OIbYkgk9EOxIwIOFQtG6QqPWwzFpsBSGCYNB/89f6W4"; __RC=4; __UF=-1; __R=1; __tb=0; cto_bundle=JhRBhV9TQUxFaHN5YUV3QnFuYTJWZzd4ZTNNRWZFR3ZpM2x1eUhaMUxnMThDNm5ZbWVRNjRIRUIlMkZqUjRma0xhZCUyRjB5OXZ3cXh5c24ycFAyaGtycjMlMkZJcU9nV1FPczlYc0FzRjh1UGRNUUVFTEpXN1dvN0poMmc1eHJFWmxwbnY4YTglMkJFNVVjV2p4YUFhZlY3VVI1RFVDemhnZyUzRCUzRA; rigelcdp-session-id=12e35a2e-3404-4c8e-b424-b38f956a48d7; __tr_geo={%22country%22:{%22name%22:%22Vietnam%22%2C%22code%22:%22VN%22}%2C%22city%22:%22Hanoi%22}; bs_onshow=1; __gads=ID=f0c111b12746b269:T=1773759966:RT=1773911093:S=ALNI_MZ13xpwkmjYH3eJCUteeTo1g-xJrA; __gpi=UID=000012218b3eacaa:T=1773759966:RT=1773911093:S=ALNI_MZhlla3_YwAsAaji90iGqIs2nj6cQ; __eoi=ID=8c86f34769843120:T=1773759966:RT=1773911093:S=AA-AfjZ-mjcPrMBtF-blIMmJOjN-; FCCDCF=%5Bnull%2Cnull%2Cnull%2Cnull%2Cnull%2Cnull%2C%5B%5B32%2C%22%5B%5C%2296c35a21-7536-4d85-88ec-c2a47483291f%5C%22%2C%5B1773759962%2C384000000%5D%5D%22%5D%5D%5D; __uif=__uid%3A4872869511907961728%7C__create%3A1771417286; FCNEC=%5B%5B%22AKsRol9a4OXZZgQLGcTIjiwwjwHXPkbTN_OXE6FQObkBibh5MA6T4JPw1DCgADTknpRk5Mqa7Tg8_HFvXMB9LcQjErDNMfWb9UVJWH8HSSQqT3wj1aWOdFAr8K6wZGQU7DrvY9v3_CYIhiK4NFFFVPNlZi51ZlWmPg%3D%3D%22%5D%5D; _ga_YCSZZTM0SE=GS2.1.s1773911086$o7$g1$t1773911112$j45$l0$h0; _ga_SXHZQVHX0C=GS2.1.s1773911086$o7$g1$t1773911112$j45$l0$h0; _ga_DSEYM6VX97=GS2.1.s1773911086$o7$g1$t1773911112$j45$l0$h0; _ga_XBF5L66EJ2=GS2.1.s1773911090$o5$g1$t1773911113$j37$l1$h1068976152'

headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/137.0.0.0 Safari/537.36",
    "Cookie": cookie_cua_chi
}

session = requests.Session(impersonate="chrome110", timeout=25)

# --- GIỮ NGUYÊN CÁC HÀM XỬ LÝ (KHÔNG ĐỔI CỘT) ---
def parse_abbr_num(text):
    if not text: return 0
    text = str(text).strip().lower().replace(",", "").replace(" ", "")
    m = re.search(r"(\d+(?:\.\d+)?)([mk]?)", text)
    if not m: return 0
    num, suffix = float(m.group(1)), m.group(2)
    if suffix == "k": return int(num * 1000)
    if suffix == "m": return int(num * 1000000)
    return int(num)

def extract_so_chuong(detail_soup, toan_bo_chu):
    for tag in detail_soup.find_all(['p', 'div', 'li', 'span']):
        txt = tag.get_text(" ", strip=True).lower()
        if txt.startswith("mới nhất:"):
            nums = re.findall(r'\d+', txt)
            if nums: return int(nums[-1])
    patterns = [r'(\d+)\s*chương\s*chính\s*văn', r'(\d+)\s*chương', r'hoàn\s*(\d+)', r'phần\s*(\d+)']
    for p in patterns:
        m = re.search(p, toan_bo_chu.lower())
        if m: return int(m.group(1))
    return 0

def get_soup(url):
    resp = session.get(url, headers=headers)
    return BeautifulSoup(resp.content, "html.parser")

def get_book_links(soup):
    links = []
    for a in soup.find_all("a", href=True):
        href = a["href"]
        if "/truyen/" in href and "khu-tim-truyen" not in href.lower():
            links.append(urljoin(BASE, href))
    return list(dict.fromkeys(links))

def find_next_page_url(soup, current_page):
    target = str(current_page + 1)
    for a in soup.find_all("a", href=True):
        if a.get_text(" ", strip=True) == target:
            return urljoin(BASE, a["href"])
    return None

# --- VÒNG LẶP QUÉT TIẾP ---
url = START_URL
data_list = []
backup_file = "Data_NgonTinh_Continue_Backup.csv"

try:
    for page in range(26, 226): # Sếp sửa số 26 nếu muốn bắt đầu từ trang khác
        print(f"\n--- ĐANG QUÉT TRANG {page} ---")
        soup = get_soup(url)
        links = get_book_links(soup)

        if not links:
            print("❌ Cookie cũ hoặc bị ban IP rồi sếp ơi!")
            break

        for book_url in links:
            try:
                time.sleep(random.uniform(3, 5)) # Tăng thời gian nghỉ để đỡ bị ban
                b_soup = get_soup(book_url)
                toan_bo_chu = b_soup.get_text(" ", strip=True)

                h_tag = b_soup.find('h2') or b_soup.find('h1')
                ten = h_tag.text.strip() if h_tag else "N/A"

                tac_gia, tinh_trang, the_loai = "Ẩn danh", "Đang cập nhật", "N/A"
                for tag_element in b_soup.find_all(['p', 'div', 'li']):
                    t_text = tag_element.get_text(" ", strip=True)
                    if t_text.startswith("Tác giả:"): tac_gia = t_text.replace("Tác giả:", "").strip()
                    elif t_text.startswith("Tình trạng:"): tinh_trang = t_text.replace("Tình trạng:", "").strip()
                    elif t_text.startswith("Thể loại:"): the_loai = t_text.replace("Thể loại:", "").strip()

                so_chuong = extract_so_chuong(b_soup, toan_bo_chu)
                stats = [tag.get_text(strip=True) for tag in b_soup.select("span[data-ready='abbrNum']")]
                vxem = parse_abbr_num(stats[0]) if len(stats) > 0 else 0
                sao = parse_abbr_num(stats[1]) if len(stats) > 1 else 0
                bluan = parse_abbr_num(stats[2]) if len(stats) > 2 else 0

                cam_on = 0
                if "Cảm ơn:" in toan_bo_chu:
                    try:
                        cam_on_text = toan_bo_chu.split("Cảm ơn:")[1].split('lần')[0].strip()
                        cam_on = parse_abbr_num(cam_on_text)
                    except: pass

                data_list.append({
                    "Trang": page, "Ten_Truyen": ten, "Tac_Gia": tac_gia,
                    "The_Loai": the_loai, "Tinh_Trang": tinh_trang, "So_Chuong": so_chuong,
                    "Luot_Xem": vxem, "Luot_Sao": sao, "Luot_Binh_Luan": bluan,
                    "Luot_Cam_On": cam_on, "Link": book_url
                })
                print(f"   [OK] {ten[:20]}")

            except: continue

        # Backup sau mỗi trang
        pd.DataFrame(data_list).to_csv(backup_file, index=False, encoding='utf-8-sig')

        next_url = find_next_page_url(soup, page)
        if not next_url: break
        url = next_url

except KeyboardInterrupt:
    print("\n🛑 Dừng theo lệnh sếp!")
finally:
    if data_list:
        final_name = f"Data_NgonTinh_Page26_End_{len(data_list)}.csv"
        pd.DataFrame(data_list).to_csv(final_name, index=False, encoding='utf-8-sig')
        files.download(final_name)

🚀 TIẾP TỤC QUÉT NGÔN TÌNH VỚI COOKIE MỚI...

--- ĐANG QUÉT TRANG 26 ---
   [OK] N/A
   [OK] N/A


In [ ]:
from curl_cffi import requests
from bs4 import BeautifulSoup
from urllib.parse import urljoin
import time
import random
import re
import pandas as pd
from google.colab import files
import os

print("🚀 KHỞI ĐỘNG LẠI TỪ TRANG 26 VỚI COOKIE MỚI...")

BASE = "https://wikicv.net"

# LINK TRANG 26 NGÔN TÌNH SẾP GỬI NÃY
START_URL = "https://wikicv.net/tim-kiem?qs=1&gender=5794f03dd7ced228f4419195&m=3&q=&start=500&so=1&y=2026&vo=1"

# COOKIE SẾP VỪA TÌM ĐƯỢC
cookie_cua_chi = '_ga=GA1.1.1694226780.1773816889; _uidcms=1773816891509758759; _ga_7NXWGWHXCV=GS2.1.s1773816889$o1$g0$t1773816892$j57$l0$h0; cto_bundle=ekqjtF9jbERIc3BSUzFnQWJhQUJYMDNDcHoyUXNDTjQ3OHNUM1NsQW8ydmFMSXF3NEElMkJJZnZzZDRaalUlMkJxbmUzVVJGTGxwOXJIdzNuNGFGSGlBWXhDS1pZV2RQSHZZTmlnQ1BOZFl4clZTM1ZKM3VoUmRuRkpkYlJERXFseERzNnYzSDAzJTJGdHdwYWpyMnVBNmg4ZWRoMXlscVElM0QlM0Q; __gads=ID=1cdf82dc5ef8f851:T=1773816894:RT=1773911614:S=ALNI_MYKkIKfP-Gda3TgYz25Q2Dlpgdp9w; __eoi=ID=9b949c42e0b4e900:T=1773816894:RT=1773911614:S=AA-AfjYX1qnBHwtDweeUtReM39I-; bs_onshow=1; rigelcdp-session-id=b530a1c7-d2b7-427a-850e-0cee24a44953; __tr_geo={%22country%22:{%22name%22:%22Vietnam%22%2C%22code%22:%22VN%22}%2C%22city%22:%22Hanoi%22}; __RC=4; express.sid="s:8RFkH4zpjICjjnlv8TxDrV1b-Mz2abSK.N3Ygy0AbDAZt9lW0Zhd2rMUxiVhG/lPwPtCCs0ObfEU"; __UF=-1; FCCDCF=%5Bnull%2Cnull%2Cnull%2Cnull%2Cnull%2Cnull%2C%5B%5B32%2C%22%5B%5C%22654178db-26af-44a8-8261-8bc28dc82753%5C%22%2C%5B1773816891%2C582000000%5D%5D%22%5D%5D%5D; __R=1; __uif=__uid%3A4241085221907960600%7C__create%3A1768914108; __tb=0; FCNEC=%5B%5B%22AKsRol9KEXhDgerms4nA75jGmIeDZ8_C_yZrbFuhAY_2_Exo6foLfZvHc4UFWNkLpo32yrv6uBGF5wEdNhSF3KOl8OdOqkheGcuMJ2jASdhJCD5XQyUNImjcdy1qUbpSH9dNqXOv4lY-Ei48I3PQs5iztZ-wvOQlEQ%3D%3D%22%5D%5D; _ga_DSEYM6VX97=GS2.1.s1773910842$o5$g1$t1773911684$j8$l0$h0; _ga_YCSZZTM0SE=GS2.1.s1773910842$o5$g1$t1773911684$j8$l0$h0; _ga_SXHZQVHX0C=GS2.1.s1773910842$o5$g1$t1773911684$j8$l0$h0; _ga_XBF5L66EJ2=GS2.1.s1773910846$o5$g1$t1773911684$j60$l1$h1883776150'

headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/137.0.0.0 Safari/537.36",
    "Cookie": cookie_cua_chi
}

session = requests.Session(impersonate="chrome110", timeout=25)

def parse_abbr_num(text):
    if not text: return 0
    text = str(text).strip().lower().replace(",", "").replace(" ", "")
    m = re.search(r"(\d+(?:\.\d+)?)([mk]?)", text)
    if not m: return 0
    num, suffix = float(m.group(1)), m.group(2)
    if suffix == "k": return int(num * 1000)
    if suffix == "m": return int(num * 1000000)
    return int(num)

def extract_so_chuong(detail_soup, toan_bo_chu):
    for tag in detail_soup.find_all(['p', 'div', 'li', 'span']):
        txt = tag.get_text(" ", strip=True).lower()
        if txt.startswith("mới nhất:"):
            nums = re.findall(r'\d+', txt)
            if nums: return int(nums[-1])
    patterns = [r'(\d+)\s*chương\s*chính\s*văn', r'(\d+)\s*chương', r'hoàn\s*(\d+)', r'phần\s*(\d+)']
    for p in patterns:
        m = re.search(p, toan_bo_chu.lower())
        if m: return int(m.group(1))
    return 0

def get_soup(url):
    resp = session.get(url, headers=headers)
    return BeautifulSoup(resp.content, "html.parser")

def get_book_links(soup):
    links = []
    for a in soup.find_all("a", href=True):
        href = a["href"]
        if "/truyen/" in href and "khu-tim-truyen" not in href.lower():
            links.append(urljoin(BASE, href))
    return list(dict.fromkeys(links))

def find_next_page_url(soup, current_page):
    target = str(current_page + 1)
    for a in soup.find_all("a", href=True):
        if a.get_text(" ", strip=True) == target:
            return urljoin(BASE, a["href"])
    return None

# --- VÒNG LẶP QUÉT ---
url = START_URL
data_list = []
backup_file = "Data_NgonTinh_T26_Backup.csv"

try:
    for page in range(26, 226):
        print(f"\n📑 Đang quét trang {page}...")
        soup = get_soup(url)
        links = get_book_links(soup)

        if not links:
            print("❌ Cookie không chạy được rồi sếp ơi (hoặc bị ban)!")
            break

        for book_url in links:
            try:
                time.sleep(random.uniform(3.0, 5.0))
                b_soup = get_soup(book_url)
                toan_bo_chu = b_soup.get_text(" ", strip=True)

                h_tag = b_soup.find('h2') or b_soup.find('h1')
                ten = h_tag.text.strip() if h_tag else "N/A"

                tac_gia, tinh_trang, the_loai = "Ẩn danh", "Đang cập nhật", "N/A"
                for tag_element in b_soup.find_all(['p', 'div', 'li']):
                    t_text = tag_element.get_text(" ", strip=True)
                    if t_text.startswith("Tác giả:"): tac_gia = t_text.replace("Tác giả:", "").strip()
                    elif t_text.startswith("Tình trạng:"): tinh_trang = t_text.replace("Tình trạng:", "").strip()
                    elif t_text.startswith("Thể loại:"): the_loai = t_text.replace("Thể loại:", "").strip()

                so_chuong = extract_so_chuong(b_soup, toan_bo_chu)
                stats = [tag.get_text(strip=True) for tag in b_soup.select("span[data-ready='abbrNum']")]
                vxem = parse_abbr_num(stats[0]) if len(stats) > 0 else 0
                sao = parse_abbr_num(stats[1]) if len(stats) > 1 else 0
                bluan = parse_abbr_num(stats[2]) if len(stats) > 2 else 0

                cam_on = 0
                if "Cảm ơn:" in toan_bo_chu:
                    try:
                        cam_on_text = toan_bo_chu.split("Cảm ơn:")[1].split('lần')[0].strip()
                        cam_on = parse_abbr_num(cam_on_text)
                    except: pass

                # Đảm bảo đủ 11 CỘT NHƯ CŨ
                data_list.append({
                    "Trang": page, "Ten_Truyen": ten, "Tac_Gia": tac_gia,
                    "The_Loai": the_loai, "Tinh_Trang": tinh_trang, "So_Chuong": so_chuong,
                    "Luot_Xem": vxem, "Luot_Sao": sao, "Luot_Binh_Luan": bluan,
                    "Luot_Cam_On": cam_on, "Link": book_url
                })
                print(f"✅ Đã quét xong: {ten[:25]}...")

            except: continue

        # Sao lưu sau mỗi trang
        pd.DataFrame(data_list).to_csv(backup_file, index=False, encoding='utf-8-sig')

        next_url = find_next_page_url(soup, page)
        if not next_url: break
        url = next_url

except KeyboardInterrupt:
    print("\n👋 Đã dừng quét theo lệnh sếp!")
finally:
    if data_list:
        final_name = f"Data_NgonTinh_T26_{len(data_list)}.csv"
        pd.DataFrame(data_list).to_csv(final_name, index=False, encoding='utf-8-sig')
        files.download(final_name)

🚀 KHỞI ĐỘNG LẠI TỪ TRANG 26 VỚI COOKIE MỚI...

📑 Đang quét trang 26...
✅ Đã quét xong: N/A...
✅ Đã quét xong: N/A...
✅ Đã quét xong: N/A...
✅ Đã quét xong: N/A...


In [ ]:
import pandas as pd
import glob
from google.colab import files

print("🚀 BẮT ĐẦU GỘP DATA...")

# 1. Tìm tất cả các file data (nếu file của sếp đuôi .xlsx thì đổi "*.csv" thành "*.xlsx")
danh_sach_file = glob.glob("*.csv")
print(f"📂 Đã tìm thấy {len(danh_sach_file)} file: {danh_sach_file}")

if len(danh_sach_file) == 0:
    print("⚠️ Sếp ơi, chưa có file nào trong Colab, sếp nhớ bấm Upload file lên trước nhé!")
else:
    # 2. Đọc và gộp tất cả các file lại
    danh_sach_dataframe = []
    for file in danh_sach_file:
        try:
            # Đọc file (nếu là excel thì dùng pd.read_excel(file))
            df = pd.read_csv(file)
            danh_sach_dataframe.append(df)
            print(f"   ✅ Đã đọc xong file: {file} ({len(df)} dòng)")
        except Exception as e:
            print(f"   ❌ Lỗi khi đọc file {file}: {e}")

    # 3. Gộp thành 1 bảng duy nhất
    df_tong = pd.concat(danh_sach_dataframe, ignore_index=True)
    print(f"\n📊 Tổng số truyện ban đầu khi gộp: {len(df_tong)}")

    # 4. Xóa các truyện bị trùng lặp (Dựa vào Link truyện để kiểm tra)
    # Bước này cực kỳ quan trọng để mô hình AI học không bị sai lệch
    df_tong = df_tong.drop_duplicates(subset=['Link'])
    print(f"🧹 Tổng số truyện sau khi lọc trùng lặp: {len(df_tong)} truyện")

    # 5. Lưu ra file tổng và tải về
    ten_file_tong = f"Data_Tong_Hop_{len(df_tong)}_Truyen.csv"
    df_tong.to_csv(ten_file_tong, index=False, encoding='utf-8-sig')

    print(f"\n🎉 Đã gộp thành công! Đang tải file tổng [{ten_file_tong}] về máy...")
    files.download(ten_file_tong)

🚀 BẮT ĐẦU GỘP DATA...
📂 Đã tìm thấy 2 file: ['Data_NgonTinh_Xong_500_Truyen.csv', 'Data_NgonTinh_Xong_503_Truyen.csv']
   ✅ Đã đọc xong file: Data_NgonTinh_Xong_500_Truyen.csv (500 dòng)
   ✅ Đã đọc xong file: Data_NgonTinh_Xong_503_Truyen.csv (500 dòng)

📊 Tổng số truyện ban đầu khi gộp: 1000
🧹 Tổng số truyện sau khi lọc trùng lặp: 1000 truyện

🎉 Đã gộp thành công! Đang tải file tổng [Data_Tong_Hop_1000_Truyen.csv] về máy...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
import pandas as pd
from google.colab import files
import io

print("🚀 BƯỚC 1: SẾP BẤM NÚT 'CHỌN TỆP' BÊN DƯỚI VÀ CHỌN CẢ 7 FILE NHÉ...")
# Lệnh này sẽ gọi cái nút Chọn tệp ra cho sếp
uploaded = files.upload()

if len(uploaded) == 0:
    print("⚠️ Sếp chưa chọn file nào rồi, chạy lại code và chọn file nhé!")
else:
    print(f"\n📂 Đã tải lên thành công {len(uploaded)} file. Đang tiến hành gộp...")

    danh_sach_dataframe = []

    # 2. Đọc và gộp các file sếp vừa tải lên
    for ten_file in uploaded.keys():
        try:
            # Đọc dữ liệu từ file vừa upload
            df = pd.read_csv(io.BytesIO(uploaded[ten_file]))
            danh_sach_dataframe.append(df)
            print(f"   ✅ Đã đọc xong: {ten_file} ({len(df)} dòng)")
        except Exception as e:
            print(f"   ❌ Lỗi khi đọc file {ten_file}: {e}")

    # 3. Gộp thành 1 bảng duy nhất
    if danh_sach_dataframe:
        df_tong = pd.concat(danh_sach_dataframe, ignore_index=True)
        print(f"\n📊 Tổng số truyện ban đầu khi gộp 7 file: {len(df_tong)}")

        # 4. Lọc các truyện bị trùng lặp (khâu cực kỳ quan trọng cho bài Học máy)
        df_tong = df_tong.drop_duplicates(subset=['Link'])
        print(f"🧹 Tổng số truyện DUY NHẤT sau khi lọc trùng: {len(df_tong)} truyện")

        # 5. Xuất file và tải về máy
        ten_file_tong = f"Data_Tong_Hop_{len(df_tong)}_Truyen.csv"
        df_tong.to_csv(ten_file_tong, index=False, encoding='utf-8-sig')

        print(f"\n🎉 ĐÃ GỘP XONG! Hệ thống đang tự động tải file [{ten_file_tong}] về máy sếp...")
        files.download(ten_file_tong)

🚀 BƯỚC 1: SẾP BẤM NÚT 'CHỌN TỆP' BÊN DƯỚI VÀ CHỌN CẢ 7 FILE NHÉ...


Saving Data_NgonTinh_Xong_500_Truyen.csv to Data_NgonTinh_Xong_500_Truyen (1).csv
Saving Data_NgonTinh_Xong_503_Truyen.csv to Data_NgonTinh_Xong_503_Truyen (1).csv
Saving Wikidich bach hop trang 26-50.csv to Wikidich bach hop trang 26-50.csv
Saving Wikidich bach hop trang 1-25.csv to Wikidich bach hop trang 1-25.csv
Saving Wikidichdammy trang 31-50.csv to Wikidichdammy trang 31-50.csv
Saving Wikidich trang 6-31.csv to Wikidich trang 6-31.csv
Saving wikidich trang1-5.csv to wikidich trang1-5 (1).csv

📂 Đã tải lên thành công 7 file. Đang tiến hành gộp...
   ✅ Đã đọc xong: Data_NgonTinh_Xong_500_Truyen (1).csv (500 dòng)
   ✅ Đã đọc xong: Data_NgonTinh_Xong_503_Truyen (1).csv (500 dòng)
   ✅ Đã đọc xong: Wikidich bach hop trang 26-50.csv (500 dòng)
   ✅ Đã đọc xong: Wikidich bach hop trang 1-25.csv (500 dòng)
   ✅ Đã đọc xong: Wikidichdammy trang 31-50.csv (400 dòng)
   ✅ Đã đọc xong: Wikidich trang 6-31.csv (500 dòng)
   ✅ Đã đọc xong: wikidich trang1-5 (1).csv (100 dòng)

📊 Tổng số truy

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
import pandas as pd
import glob
from google.colab import files

print("🚀 BẮT ĐẦU GỘP DATA TỰ ĐỘNG...")

# Quét tự động tất cả các file đuôi .csv đang có sẵn trong Colab
danh_sach_file = glob.glob("*.csv")

# Bỏ qua file tổng nếu sếp lỡ chạy nhiều lần
danh_sach_file = [f for f in danh_sach_file if "Data_Tong_Hop" not in f]

print(f"📂 Đã tìm thấy {len(danh_sach_file)} file: {danh_sach_file}")

if len(danh_sach_file) == 0:
    print("⚠️ Colab chưa thấy file nào! Sếp nhớ kéo thả 7 file vào biểu tượng Thư mục bên trái nhé!")
else:
    danh_sach_dataframe = []
    for file in danh_sach_file:
        try:
            df = pd.read_csv(file)
            danh_sach_dataframe.append(df)
            print(f"   ✅ Đã gom xong: {file} ({len(df)} dòng)")
        except Exception as e:
            print(f"   ❌ Lỗi: {e}")

    # Gộp và lọc trùng
    df_tong = pd.concat(danh_sach_dataframe, ignore_index=True)
    print(f"\n📊 Tổng truyện ban đầu: {len(df_tong)}")

    df_tong = df_tong.drop_duplicates(subset=['Link'])
    print(f"🧹 Tổng truyện SAU KHI LỌC TRÙNG LẶP: {len(df_tong)} truyện")

    # Lưu và tải về
    ten_file_tong = f"Data_Tong_Hop_{len(df_tong)}_Truyen.csv"
    df_tong.to_csv(ten_file_tong, index=False, encoding='utf-8-sig')

    print(f"\n🎉 ĐÃ XONG! Đang tải file [{ten_file_tong}] về máy...")
    files.download(ten_file_tong)

🚀 BẮT ĐẦU GỘP DATA TỰ ĐỘNG...
📂 Đã tìm thấy 3 file: ['Data_NgonTinh_Xong_500_Truyen.csv', 'Data_NgonTinh_Xong_503_Truyen.csv', 'wikidich trang1-5.csv']
   ✅ Đã gom xong: Data_NgonTinh_Xong_500_Truyen.csv (500 dòng)
   ✅ Đã gom xong: Data_NgonTinh_Xong_503_Truyen.csv (500 dòng)
   ✅ Đã gom xong: wikidich trang1-5.csv (100 dòng)

📊 Tổng truyện ban đầu: 1100
🧹 Tổng truyện SAU KHI LỌC TRÙNG LẶP: 1100 truyện

🎉 ĐÃ XONG! Đang tải file [Data_Tong_Hop_1100_Truyen.csv] về máy...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>